# Colab GPU — video minh hoạ + số liệu GPU (ADR-0002)

Notebook này chạy **headless** (không cần màn hình) để tạo:
- `data/out_gpu.mp4` — video đã overlay (mask + ô bao + quỹ đạo + HUD) ở tốc độ GPU thật
- `data/gpu_metrics.csv` / `data/gpu_summary.json` — số liệu từng frame + tổng hợp

> ⚠️ Số liệu trong `gpu_summary.json` là **số GPU thật** — dùng cho mục "Đánh giá hiệu năng",
> đặt cạnh số CPU laptop để đối chiếu trung thực. Không dùng số GPU để nói "demo live realtime".

**Cách dùng:** menu **Runtime → Change runtime type → Hardware accelerator = T4 GPU** → Save,
rồi chạy lần lượt các cell dưới (Shift+Enter).


In [ ]:
# 1) Kiểm tra GPU — phải thấy Tesla T4
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA sẵn sàng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("Chưa bật GPU: Runtime → Change runtime type → T4 GPU")


In [ ]:
# 2) Lấy code + SAMURAI + checkpoint (khoảng 2-4 phút)
!git clone -q https://github.com/BinhNguyen1912/DoAn-NAB.git
%cd DoAn-NAB
!git clone -q --depth 1 https://github.com/yangchris11/samurai.git samurai
!mkdir -p samurai/sam2/checkpoints
!wget -q -O samurai/sam2/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092823/sam2.1_hiera_tiny.pt
!ls -lh samurai/sam2/checkpoints/

# Colab ĐÃ có numpy/opencv/pillow/torch-CUDA. KHÔNG cài lại numpy/opencv để tránh
# làm hỏng môi trường Colab (pip sẽ đòi restart runtime). Chỉ cài phần còn thiếu:
!pip install -q pyyaml
!pip install -q -e samurai/sam2
import cv2, numpy, yaml, PIL
print("opencv", cv2.__version__, "| numpy", numpy.__version__)
print("cài đặt xong")


## 3) Chuẩn bị video đầu vào

**Cách A (khuyến nghị):** quay video bằng điện thoại (10–15 giây, người/vật chuyển động),
rồi kéo file vào panel **Files** bên trái (thư mục `/content/DoAn-NAB/data/`), đổi tên `input.mp4`.

**Cách B:** chạy cell dưới để sinh video test tổng hợp (ô vuông di chuyển) — dùng để kiểm tra
notebook chạy được trước khi có video thật.


In [ ]:
# 3b) Sinh video test nếu chưa có input.mp4 (ô vuông cam di chuyển, 60 frame)
import os
if not os.path.exists("data/input.mp4"):
    from src.synthetic_video import generate_tracking_sequence
    import cv2
    fx = generate_tracking_sequence(num_frames=60, frame_size=(720, 1280), square_size=120,
                                    start=(200, 200), velocity=(14, 6))
    out = cv2.VideoWriter("data/input.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 25, (1280, 720))
    for f in fx.frames:
        out.write(f)
    out.release()
    print("đã sinh data/input.mp4 (ô vuông tại frame 0:", fx.bboxes[0], ")")
else:
    print("đã có data/input.mp4")
import cv2
cap = cv2.VideoCapture("data/input.mp4")
print("video:", int(cap.get(3)), "x", int(cap.get(4)), "|", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), "frame")
cap.release()


## 4) Chạy pipeline trên GPU

`--bbox` là ô chọn đối tượng ở **frame 0** (x,y,w,h). Với video tự sinh ở trên dùng `200,200,120,120`;
với video thật, xem frame 0 để chọn toạ độ (hoặc bỏ `--bbox` để dùng ô giữa khung).


In [ ]:
!python tools/run_live.py --source data/input.mp4 --bbox "200,200,120,120" --device cuda --record data/out_gpu.mp4 --metrics-csv data/gpu_metrics.csv --metrics-summary data/gpu_summary.json


In [ ]:
# 5) Xem số liệu + video kết quả ngay trong notebook
import json
from IPython.display import Video, display
s = json.load(open("data/gpu_summary.json"))
print("=== SỐ LIỆU GPU (thật) ===")
for k, v in s.items():
    print(f"  {k:20} {round(v,3) if isinstance(v, float) else v}")
display(Video("data/out_gpu.mp4", embed=True, width=720))


In [ ]:
# 6) Tải file về máy để đưa vào báo cáo
from google.colab import files
for name in ("data/out_gpu.mp4", "data/gpu_metrics.csv", "data/gpu_summary.json"):
    files.download(name)


## 7) Ghi chú cho báo cáo

- Đặt **số GPU** (cell 5) cạnh **số CPU laptop** (`data/webcam_summary.json` / bảng trong README
  mục "Đánh giá hiệu năng") — nêu rõ phần cứng từng bên.
- Câu trả lời cho hội đồng: *hệ thống real-time-capable trên GPU (số đo T4); trên laptop CPU
  chỉ đạt ~0.2–2 FPS nên demo live là slideshow có chủ đích (ADR-0002).*
- Nếu muốn video mượt hơn cho slide: quay video nguồn 10–15 giây, chạy lại cell 4 — GPU xử lý
  ~15–25 FPS, thời gian chạy chỉ khoảng 20–30 giây.
